In [1]:
.venv

SyntaxError: invalid syntax (1351572534.py, line 1)

In [ ]:
import pandas as pd

# =========================================================
# PROJECT PATH
# =========================================================

base = r"C:\Users\satwi\OneDrive\Desktop\PROJECTS\Urban_Mobility_Risk_Intelligence\processed"

# =========================================================
# FILE PATHS
# =========================================================

master_file = base + r"\bengaluru_crashes_station_master.csv"
risk_file = base + r"\station_risk_score.csv"
profile_file = base + r"\station_risk_profile.csv"
geo_file = base + r"\station_risk_geocoded_final.csv"
priority_file = base + r"\final_intervention_priority_ranking.csv"

# =========================================================
# LOAD DATASETS
# =========================================================

master = pd.read_csv(master_file)
risk = pd.read_csv(risk_file)
profile = pd.read_csv(profile_file)
geo = pd.read_csv(geo_file)
priority = pd.read_csv(priority_file)

print("All datasets loaded successfully!")

print("\nMaster:", master.shape)
print("Risk:", risk.shape)
print("Profile:", profile.shape)
print("Geocoded:", geo.shape)
print("Priority:", priority.shape)

All datasets loaded successfully!

Master: (371, 9)
Risk: (60, 29)
Profile: (60, 17)
Geocoded: (60, 17)
Priority: (37, 9)


In [ ]:
# =========================================================
# CREATE STATION-LEVEL DASHBOARD TABLE
# =========================================================

station_data = risk.copy()

# ---------------------------------------------------------
# Add geographic coordinates
# ---------------------------------------------------------

geo_columns = [
    "station",
    "latitude",
    "longitude",
    "location_status"
]

station_data = station_data.merge(
    geo[geo_columns],
    on="station",
    how="left"
)

# ---------------------------------------------------------
# Add final intervention priority
# ---------------------------------------------------------

priority_columns = [
    "station",
    "priority_rank",
    "priority_index",
    "priority_category"
]

station_data = station_data.merge(
    priority[priority_columns],
    on="station",
    how="left"
)

# ---------------------------------------------------------
# Add trend information
# ---------------------------------------------------------

profile_columns = [
    "station",
    "trend_category",
    "priority_flag"
]

station_data = station_data.merge(
    profile[profile_columns],
    on="station",
    how="left"
)

# =========================================================
# CREATE YEAR-LEVEL DASHBOARD TABLE
# =========================================================

yearly = master.groupby("year").agg(
    total_crashes=("total_crashes", "sum"),
    fatal_crashes=("fatal_crashes", "sum"),
    nonfatal_crashes=("nonfatal_crashes", "sum")
).reset_index()

# Preserve missing killed/injured values for 2024 and 2025
killed = master.groupby("year")["people_killed"].sum(min_count=1)
injured = master.groupby("year")["people_injured"].sum(min_count=1)

yearly["people_killed"] = yearly["year"].map(killed)
yearly["people_injured"] = yearly["year"].map(injured)

# =========================================================
# SAVE DASHBOARD DATA
# =========================================================

station_output = base + r"\dashboard_station_master.csv"
yearly_output = base + r"\dashboard_yearly_trends.csv"

station_data.to_csv(
    station_output,
    index=False
)

yearly.to_csv(
    yearly_output,
    index=False
)

# =========================================================
# VALIDATION
# =========================================================

print("=" * 80)
print("DASHBOARD DATA PREPARATION COMPLETE")
print("=" * 80)

print("\nStation-level table:", station_data.shape)
print("Unique stations:", station_data["station"].nunique())

print("\nYear-level table:", yearly.shape)
print("Years:", yearly["year"].min(), "to", yearly["year"].max())

print("\nStation dashboard columns:")
print(station_data.columns.tolist())

print("\nYearly dashboard columns:")
print(yearly.columns.tolist())

print("\nFiles saved:")
print(station_output)
print(yearly_output)

DASHBOARD DATA PREPARATION COMPLETE

Station-level table: (60, 37)
Unique stations: 60

Year-level table: (8, 6)
Years: 2018 to 2025

Station dashboard columns:
['station', 'years_available', 'first_year', 'last_year', 'total_crashes', 'total_fatal_crashes', 'total_people_killed', 'total_people_injured', 'avg_annual_crashes', 'avg_annual_fatal_crashes', 'fatal_crash_pct', 'recent_years_available', 'recent_total_crashes', 'recent_fatal_crashes', 'recent_avg_annual_crashes', 'recent_avg_annual_fatal_crashes', 'recent_fatal_crash_pct', 'crash_trend_per_year', 'recent_vs_historical_pct', 'crash_frequency_score', 'fatality_score', 'severity_score', 'recent_crash_score', 'recent_fatality_score', 'risk_score', 'risk_category', 'data_coverage', 'recent_data_status', 'risk_rank', 'latitude', 'longitude', 'location_status', 'priority_rank', 'priority_index', 'priority_category', 'trend_category', 'priority_flag']

Yearly dashboard columns:
['year', 'total_crashes', 'fatal_crashes', 'nonfatal_cra